In [ ]:
dataset = "eco2mix-national-tr"
jours_glissants = 3
date_debut = ""          # vide = aujourd'hui - jours_glissants ; historique : "2024-01-01" avec eco2mix-national-cons-def

In [ ]:
import json, os, requests
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F

TZ = ZoneInfo("Europe/Paris")
run_ts = datetime.now(timezone.utc)

if not date_debut:
    date_debut = (datetime.now(TZ).date() - timedelta(days=int(jours_glissants))).isoformat()

url = f"https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/{dataset}/exports/json"
params = {
    "where": f"date_heure >= date'{date_debut}'",
    "timezone": "Europe/Paris",
}

r = requests.get(url, params=params, timeout=300)
r.raise_for_status()
records = r.json()
print(f"{len(records)} lignes reçues pour {dataset} depuis le {date_debut}")

if not records:
    notebookutils.notebook.exit("AUCUNE_DONNEE")

In [ ]:
rel = f"Files/bronze/eco2mix/{dataset}/ingest_date={run_ts:%Y-%m-%d}/{run_ts:%H%M%S}.json"
local = f"/lakehouse/default/{rel}"
os.makedirs(os.path.dirname(local), exist_ok=True)
with open(local, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False)
print("Fichier écrit :", rel)

In [ ]:
df = (
    spark.read.option("multiLine", True).json(rel)
    .withColumn("_ingested_at", F.lit(run_ts.strftime("%Y-%m-%d %H:%M:%S")).cast("timestamp"))
    .withColumn("_source_file", F.lit(rel))
    .withColumn("_dataset", F.lit(dataset))
)
# Bronze = brut : toutes les colonnes de données en texte, le typage se fait dans le silver
donnees = [c for c in df.columns if not c.startswith("_")]
df = df.select(*[F.col(c).cast("string").alias(c) for c in donnees],
               "_ingested_at", "_source_file", "_dataset")
df.printSchema()

(df.write.mode("append")
   .option("mergeSchema", "true")
   .saveAsTable("bronze_eco2mix"))

notebookutils.notebook.exit(rel)